# cbj CatBoost T4×2 · 공통 5-fold 평가
CatBoost만 두 GPU로 실제 학습합니다. LightGBM과 XGBoost는 이번 실행에서 제외합니다. 기존 group fold·F0·300 rounds는 유지하며 파생변수 추가와 상세 exact/site는 끕니다. 전체 OOF Macro-F1과 학습 시간을 보고합니다. GPU 연산은 같은 seed에도 미세한 비결정성이 있을 수 있습니다. 대회 제출은 실행하지 않습니다.

In [ ]:
# All notebook imports are collected here.
from pathlib import Path
import json
import subprocess
import sys
from IPython.display import display

# Run controls: change here, not in result cells.
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path.cwd() / 'cbj_catboost_gpu_run'
CONFIG_FILENAME = 'cbj_catboost_gpu_config.json'
RUN_MODE = 'evaluate'  # 'evaluate' runs the configured real-data experiments
MAKE_SUBMISSION = False  # generates CSV only after internal gates pass; never submits to a competition

configs = list(INPUT_ROOT.rglob(CONFIG_FILENAME))
assert len(configs) == 1, 'Ambiguous or missing config input'
CONFIG_PATH = configs[0]
CODE_PATH = CONFIG_PATH.parent / 'cbj_tree_experiments.py'
assert CODE_PATH.exists()


## CatBoost GPU 설정과 실행
실제 T4 2장이 감지되지 않으면 중단합니다. devices=0:1로 두 GPU를 지정하며 CPU로 자동 전환하지 않습니다.

In [ ]:
config = json.loads(CONFIG_PATH.read_text())
display({'mode': RUN_MODE, 'features': config['features'], 'experiments': config['experiments'], 'selection': config['selection']})
command = [sys.executable, str(CODE_PATH), '--config', str(CONFIG_PATH), '--input-root', str(INPUT_ROOT), '--output', str(OUTPUT_ROOT), '--mode', RUN_MODE]
if MAKE_SUBMISSION:
    command.append('--make-submission')
subprocess.run(command, check=True)


## 공통 OOF 평가 결과
앵커와의 거리는 보조 진단이며 점수 통과 기준에 사용하지 않습니다. 이번 실행에서는 test 예측·제출 CSV 생성을 끕니다.

In [ ]:
result_file = OUTPUT_ROOT / ('validation.json' if RUN_MODE == 'validate' else 'selection.json')
result = json.loads(result_file.read_text())
display({key: value for key, value in result.items() if key != 'config'})
print('Competition submission: not performed')
